<b>Quiz 3 - Problem 2</b>

In [17]:
# data

import csv
f = open("quiz 3 problem 2.csv")
csvfile = csv.DictReader(f, delimiter=',')
headers = csvfile.fieldnames

table = []
for row in csvfile:
    table.append(row)
    
f.close()

# Create set of Nodes.
Nodes = set()
# Create dictionary "Distance" for each arc.
# We can borrow the keys from this dictionary to define the arc set of the network.
Distance = {}
for row in table:
    fromNode = row['From']
    toNode = row['To']
    Distance[(fromNode,toNode)] = float(row['Distance (miles)'])
    Nodes.add(fromNode)
    Nodes.add(toNode)

Arcs = Distance.keys()

In [18]:
Nodes

{'1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9', 'P'}

In [19]:
Distance

{('1', '3'): 80.0,
 ('2', '4'): 68.0,
 ('3', '4'): 54.0,
 ('3', '5'): 94.0,
 ('3', '6'): 91.0,
 ('4', '3'): 51.0,
 ('4', '6'): 57.0,
 ('4', '7'): 116.0,
 ('5', '8'): 17.0,
 ('6', '9'): 15.0,
 ('7', '10'): 13.0,
 ('8', '11'): 71.0,
 ('8', '12'): 139.0,
 ('9', '11'): 92.0,
 ('9', '12'): 74.0,
 ('10', '12'): 83.0,
 ('11', 'P'): 62.0,
 ('12', 'P'): 47.0}

In [20]:
Arcs

dict_keys([('1', '3'), ('2', '4'), ('3', '4'), ('3', '5'), ('3', '6'), ('4', '3'), ('4', '6'), ('4', '7'), ('5', '8'), ('6', '9'), ('7', '10'), ('8', '11'), ('8', '12'), ('9', '11'), ('9', '12'), ('10', '12'), ('11', 'P'), ('12', 'P')])

In [21]:
Shipment = {'S1', 'S2'}
Origin = { 'S1' : '1', 'S2' : '2' }
Destination = { 'S1' : 'P', 'S2' : 'P' }
Volume = { 'S1' :1 , 'S2' :1}

In [22]:
# define supply/demand for each node and each commodity
Supply = {}
for i in Nodes:
    
    for k in Shipment:
        if i==Origin[k]:
            Supply[i,k] = -1 * Volume[k]
        elif i==Destination[k]:
            Supply[i,k] = Volume[k]
        else:
            Supply[i,k] = 0

In [23]:
Supply

{('3', 'S1'): 0,
 ('3', 'S2'): 0,
 ('2', 'S1'): 0,
 ('2', 'S2'): -1,
 ('6', 'S1'): 0,
 ('6', 'S2'): 0,
 ('8', 'S1'): 0,
 ('8', 'S2'): 0,
 ('9', 'S1'): 0,
 ('9', 'S2'): 0,
 ('11', 'S1'): 0,
 ('11', 'S2'): 0,
 ('4', 'S1'): 0,
 ('4', 'S2'): 0,
 ('12', 'S1'): 0,
 ('12', 'S2'): 0,
 ('7', 'S1'): 0,
 ('7', 'S2'): 0,
 ('1', 'S1'): -1,
 ('1', 'S2'): 0,
 ('P', 'S1'): 1,
 ('P', 'S2'): 1,
 ('5', 'S1'): 0,
 ('5', 'S2'): 0,
 ('10', 'S1'): 0,
 ('10', 'S2'): 0}

In [24]:
# define a set of arc/commodity pairs (useful for variables)
ArcShipment = { (i,j,k) for (i,j) in Arcs for k in Shipment }

In [25]:
from docplex.mp.model import Model
mdl = Model()

In [26]:
shipped = mdl.continuous_var_dict(ArcShipment, lb=0, name='shipped')

In [27]:
# objective
mdl.minimize(mdl.sum(Distance[i,j]*shipped[i,j,k] for (i,j,k) in shipped))

In [28]:
# flow balance constraints for each commodity and for each node
for k in Shipment:
    for j in Nodes:
        inflow = sum(shipped[i,j,k] for i in Nodes if (i,j) in Arcs)
        outflow = sum(shipped[j,i,k] for i in Nodes if (j,i) in Arcs)
        mdl.add_constraint(inflow - outflow == Supply[j,k])

In [29]:
Bridge_Tunnel = {('5', '8'), ('6', '9'), ('7', '10')}
for arc in Bridge_Tunnel:
    if arc == ('5', '8'):
        mdl.add_constraint(mdl.sum(shipped[i,j,k] for k in Shipment for i, j in Arcs if (i, j) == arc) == 0)
    else:
        mdl.add_constraint(mdl.sum(shipped[i,j,k] for k in Shipment for i, j in Arcs if (i, j) == arc) <= 1)

In [30]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.00370216,status='optimal')

In [31]:
mdl.print_solution()

objective: 634.000
status: OPTIMAL_SOLUTION(2)
  shipped_12_P_S2=1.000
  shipped_12_P_S1=1.000
  shipped_1_3_S1=1.000
  shipped_6_9_S1=1.000
  shipped_3_6_S1=1.000
  shipped_4_7_S2=1.000
  shipped_2_4_S2=1.000
  shipped_7_10_S2=1.000
  shipped_10_12_S2=1.000
  shipped_9_12_S1=1.000
